# Novelty Analysis — ECE, CAG, and Grad-CAM Shift

Reads raw JSON from notebooks 01-03 and computes:
- ECE (Expected Calibration Error): per model per dataset
- CAG (Consistency-Accuracy Gap): Overconfident vs Consistently Wrong
- Grad-CAM cosine shift for stable vs flipped predictions

**Prereq:** run notebooks 01-03 first.

In [ ]:
BASE_CIFAR  = r'E:\decision_consistency_cifar_outputs\tables'
BASE_STL10  = r'E:\decision_consistency_stl10_outputs\tables'
BASE_INTEL  = r'E:\decision_consistency_intel_outputs\tables'
OUTPUT_DIR  = r'E:\decision_consistency_novelty'

In [ ]:
import os, json, csv
import numpy as np
from scipy.stats import pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.makedirs(OUTPUT_DIR,exist_ok=True)
datasets_info={'CIFAR-10':BASE_CIFAR,'STL-10':BASE_STL10,'Intel-Scene':BASE_INTEL}

In [ ]:
all_data={}
for ds_name,base in datasets_info.items():
    all_data[ds_name]={}
    if not os.path.isdir(base): print(f'WARNING: {base} not found'); continue
    for fname in os.listdir(base):
        if not (fname.startswith('raw_') and fname.endswith('.json')): continue
        with open(os.path.join(base,fname)) as f: samples=json.load(f)
        parts=fname.replace('raw_','').replace('.json','').split('_')
        model_raw='_'.join(parts[2:]); model_name=model_raw.replace('ResNet_18','ResNet-18').replace('ResNet_50','ResNet-50').replace('VGG_16','VGG-16')
        all_data[ds_name][model_name]=samples
print('Loaded:',[f'{ds}: {list(m.keys())}' for ds,m in all_data.items()])

In [ ]:
def compute_ece(confs,corrects,n_bins=10):
    bins=np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        mask=(confs>=bins[i])&(confs<bins[i+1])
        if mask.sum()==0: continue
        ece+=np.abs(np.mean(confs[mask])-np.mean(corrects[mask]))*(mask.sum()/len(confs))
    return ece
results_table=[]
for ds_name in all_data:
    for model_name,samples in all_data[ds_name].items():
        true_labels=np.array([s['true_label'] for s in samples]); preds=np.array([s['orig_pred'] for s in samples]); confs=np.array([s['orig_conf'] for s in samples]); cs_arr=np.array([s.get('Consistency_Score',0) for s in samples])
        accuracy=np.mean(preds==true_labels); ece=compute_ece(confs,preds==true_labels); avg_cs=np.mean(cs_arr); cag=accuracy-avg_cs
        results_table.append({'Dataset':ds_name,'Model':model_name,'Accuracy':round(accuracy,4),'ECE':round(ece,4),'Avg_CS':round(avg_cs,4),'CAG':round(cag,4)})
csv_path=os.path.join(OUTPUT_DIR,'accuracy_ece_cag.csv')
with open(csv_path,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['Dataset','Model','Accuracy','ECE','Avg_CS','CAG']); w.writeheader(); w.writerows(results_table)
print('Saved:',csv_path)
for r in sorted(results_table,key=lambda x:x['Dataset']):
    print(f"{r['Dataset']:<12} {r['Model']:<14} Acc={r['Accuracy']:.3f} ECE={r['ECE']:.3f} CS={r['Avg_CS']:.3f} CAG={r['CAG']:.4f}")

In [ ]:
if results_table:
    ece_vals=[r['ECE'] for r in results_table]; cs_vals=[r['Avg_CS'] for r in results_table]
    r_val,p_val=pearsonr(ece_vals,cs_vals)
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    axes[0].scatter(ece_vals,cs_vals,color='steelblue',s=70)
    for row in results_table: axes[0].annotate(row['Model'][:6],(row['ECE'],row['Avg_CS']),fontsize=8)
    axes[0].set_xlabel('ECE'); axes[0].set_ylabel('CS'); axes[0].set_title(f'ECE vs CS  r={r_val:.2f} p={p_val:.3f}'); axes[0].grid(alpha=0.3)
    sorted_r=sorted(results_table,key=lambda x:x['CAG']); labels=[f"{r['Dataset'][:4]} {r['Model'][:6]}" for r in sorted_r]; cag_v=[r['CAG'] for r in sorted_r]
    axes[1].bar(labels,cag_v,color=['tomato' if v>0 else 'steelblue' for v in cag_v]); axes[1].axhline(0,color='black',lw=1.5); axes[1].set_ylabel('CAG'); axes[1].tick_params(axis='x',rotation=45); axes[1].set_title('CAG Dichotomy'); axes[1].grid(axis='y',alpha=0.3)
    plt.tight_layout(); path=os.path.join(OUTPUT_DIR,'ece_vs_cs_and_cag.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
# Grad-CAM cosine similarity: stable vs flipped (STL-10 quantitative results from notebook 02)
cam_data={'ResNet-18':(0.991,0.922),'ResNet-50':(0.986,0.883),'VGG-16':(0.953,0.832),'MobileNetV2':(0.991,0.954)}
model_names=list(cam_data.keys()); stable_vals=[cam_data[m][0] for m in model_names]; flipped_vals=[cam_data[m][1] for m in model_names]
x=np.arange(len(model_names)); w=0.35
fig,ax=plt.subplots(figsize=(8,5))
ax.bar(x-w/2,stable_vals,w,label='Label Stable',color='steelblue'); ax.bar(x+w/2,flipped_vals,w,label='Label Flipped',color='tomato')
ax.set_xticks(x); ax.set_xticklabels(model_names); ax.set_ylim(0.80,1.0); ax.set_ylabel('Mean CAM Cosine Similarity'); ax.set_title('Grad-CAM Shift: Stable vs Flipped — STL-10'); ax.legend(); ax.grid(axis='y',alpha=0.3); plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'gradcam_shift.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)